# GNNocRoute-DRL: PPO Training — Phase 2
## GPU-Accelerated Training + BookSim2 Validation

**Input:** Dense BookSim2 data (378 configs × 21 IR × 3 topo × 3 traffic)

**Output:** Trained PPO agent + routing policy

---
**Instructions:** Runtime → Run all (15-20 phút)

In [ ]:
# === SETUP ===
import os, sys, json, time, random, subprocess, tempfile, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from google.colab import drive

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/GNNocRoute_DRL_Results/phase2'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# === GNN ENCODER ===
class GNNEncoder(nn.Module):
    def __init__(self, in_dim=4, hidden=64, out=32, heads=4):
        super().__init__()
        from torch_geometric.nn import GATv2Conv
        self.conv1 = GATv2Conv(in_dim, hidden//heads, heads=heads)
        self.conv2 = GATv2Conv(hidden, out, heads=1)
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(out)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index); x = self.norm1(x); x = F.elu(x)
        x = self.conv2(x, edge_index); x = self.norm2(x); return x

class PPOPolicy(nn.Module):
    """PPO actor-critic network."""
    def __init__(self, state_dim=32):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim+4, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 4))  # 4 actions: XY, YX (2) + probability
        self.critic = nn.Sequential(
            nn.Linear(state_dim+4, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1))
    def forward(self, x):
        return self.actor(x), self.critic(x)

print('✅ Models ready')

In [ ]:
# === GRAPH BUILDER ===
def build_mesh_edges(G):
    N = G*G; edges = []
    for y in range(G):
        for x in range(G):
            i = y*G+x
            if x>0: edges.append([i, y*G+(x-1)])
            if x<G-1: edges.append([i, y*G+(x+1)])
            if y>0: edges.append([i, (y-1)*G+x])
            if y<G-1: edges.append([i, (y+1)*G+x])
    return torch.LongTensor(edges).t().to(DEVICE)

In [ ]:
# === PPO TRAINING ===
def train_ppo(gnn, policy, edge_index, traffic='hotspot', topology='mesh44',
              episodes=500, num_nodes=16):
    """Train PPO agent on NoC routing task."""
    opt = torch.optim.Adam(list(gnn.parameters()) + list(policy.parameters()), lr=3e-4)
    G = int(np.sqrt(num_nodes))
    
    rewards = []
    t0 = time.time()
    
    # Pre-train GNN
    pt_opt = torch.optim.Adam(gnn.parameters(), lr=1e-3)
    for ep in range(200):
        x = torch.randn(num_nodes, 4).to(DEVICE)
        emb = gnn(x, edge_index)
        loss = -torch.var(emb) + 0.01 * torch.norm(emb)
        pt_opt.zero_grad(); loss.backward(); pt_opt.step()
    print(f'  GNN pre-trained')
    
    # PPO training
    for ep in range(episodes):
        # Generate random congestion state
        congestion = np.random.uniform(0, 1, (num_nodes,))
        
        # Build state
        x = np.zeros((num_nodes, 4))
        x[:, 0] = congestion / max(np.max(congestion), 1)
        x[:, 1] = np.convolve(congestion, np.ones(G)/G, mode='same')
        x[:, 3] = np.arange(num_nodes) / num_nodes
        
        state = torch.FloatTensor(x).to(DEVICE)
        
        # GNN forward
        with torch.no_grad():
            embeddings = gnn(state, edge_index)  # (N, 32)
        
        # For each (src, dst), compute routing decision
        total_reward = 0
        for src in range(min(num_nodes, 8)):  # sample for speed
            for dst in range(num_nodes):
                if src == dst: continue
                
                # Features: src_emb + dst_emb + congestion + distance
                src_emb = embeddings[src]
                dst_emb = embeddings[dst]
                src_cong = torch.tensor(congestion[src]).to(DEVICE)
                dst_cong = torch.tensor(congestion[dst]).to(DEVICE)
                dist = abs(src%G - dst%G) + abs(src//G - dst//G)
                dist = torch.tensor(dist / (2*G)).to(DEVICE)
                
                features = torch.cat([src_emb, dst_emb, 
                    src_cong.unsqueeze(0), dst_cong.unsqueeze(0), 
                    dist.unsqueeze(0), 
                    (src_cong - dst_cong).unsqueeze(0)]).to(DEVICE)
                
                # Forward
                logits, value = policy(features.unsqueeze(0))
                probs = F.softmax(logits, dim=-1)
                dist = torch.distributions.Categorical(probs)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                
                # Reward: 0=XY (lower congestion), 1=YX (lower congestion)
                # Choose the path that avoids congestion
                if action == 0:  # XY
                    reward = -congestion[src] * 0.3 - congestion[dst] * 0.7
                else:  # YX
                    reward = -congestion[dst] * 0.3 - congestion[src] * 0.7
                
                total_reward += reward
                
                # PPO update (simplified)
                advantage = torch.tensor(reward - value.item()).to(DEVICE)
                actor_loss = -log_prob * advantage.detach()
                critic_loss = F.mse_loss(value.squeeze(), 
                    torch.tensor(reward).to(DEVICE).unsqueeze(0))
                loss = actor_loss + 0.5 * critic_loss
                
                opt.zero_grad(); loss.backward(); opt.step()
        
        rewards.append(total_reward)
        if (ep+1) % 100 == 0:
            print(f'  Ep {ep+1}/{episodes} | R={np.mean(rewards[-50:]):.2f}')
    
    print(f'  ✅ Done in {(time.time()-t0)/60:.1f} min | Avg R={np.mean(rewards[-50:]):.2f}')
    return rewards

print('✅ Training function ready')

In [ ]:
# === MAIN TRAINING LOOP ===
print('=' * 60)
print('GNNocRoute-DRL: Phase 2 — PPO Training')
print('=' * 60)

configs = [
    ('mesh44', 16, 'hotspot', 500),
    ('mesh44', 16, 'uniform', 300),
    ('mesh88', 64, 'hotspot', 300),
    ('mesh1616', 256, 'hotspot', 200),
]

all_results = {}
for topo_name, num_nodes, traffic, episodes in configs:
    print(f'\n{"="*50}')
    print(f'Training: {topo_name} | {traffic} | {episodes} eps')
    print(f'{"="*50}')
    
    edge_index = build_mesh_edges(int(np.sqrt(num_nodes)))
    gnn = GNNEncoder().to(DEVICE)
    policy = PPOPolicy().to(DEVICE)
    
    rewards = train_ppo(gnn, policy, edge_index, traffic, topo_name, episodes, num_nodes)
    
    # Save
    torch.save({'gnn': gnn.state_dict(), 'policy': policy.state_dict()},
               f'{OUT}/ppo_{topo_name}_{traffic}.pt')
    
    all_results[f'{topo_name}_{traffic}'] = {
        'avg_reward': float(np.mean(rewards[-50:])),
        'episodes': episodes,
        'device': DEVICE
    }
    print(f'  ✅ Saved to {OUT}/ppo_{topo_name}_{traffic}.pt')

json.dump(all_results, open(f'{OUT}/training_results.json', 'w'), indent=2)
print(f'\n✅ Phase 2 complete! Results saved to Drive')

In [ ]:
# === GENERATE ROUTING TABLES ===
print('=== Generating Routing Tables for BookSim2 ===')

for topo_name, num_nodes, traffic, _ in configs:
    model_path = f'{OUT}/ppo_{topo_name}_{traffic}.pt'
    if not os.path.exists(model_path):
        print(f'  Skip {topo_name}: no model')
        continue
    
    G = int(np.sqrt(num_nodes))
    edge_index = build_mesh_edges(G)
    gnn = GNNEncoder().to(DEVICE)
    policy = PPOPolicy().to(DEVICE)
    ckpt = torch.load(model_path, map_location=DEVICE)
    gnn.load_state_dict(ckpt['gnn'])
    policy.load_state_dict(ckpt['policy'])
    
    # Compute routing table
    x = torch.zeros(num_nodes, 4).to(DEVICE)
    x[:, 3] = torch.arange(num_nodes).float().to(DEVICE) / num_nodes
    with torch.no_grad():
        embeddings = gnn(x, edge_index)
    
    table = np.zeros((num_nodes, num_nodes), dtype=int)
    for src in range(num_nodes):
        for dst in range(num_nodes):
            if src == dst: continue
            src_emb = embeddings[src]
            dst_emb = embeddings[dst]
            features = torch.cat([src_emb, dst_emb, 
                torch.zeros(2).to(DEVICE), torch.zeros(2).to(DEVICE)]).unsqueeze(0)
            logits, _ = policy(features)
            action = logits.argmax().item()
            table[src, dst] = 1 if action >= 2 else 0
    
    # Save
    hdr = f'// PPO routing table for {topo_name}/{traffic}\n'
    hdr += f'static const int ppo_table_{topo_name}_{traffic}[{num_nodes}][{num_nodes}] = {{\n'
    for src in range(num_nodes):
        row = ','.join(str(table[src, d]) for d in range(num_nodes))
        hdr += f'  {{{row}{"," if src < num_nodes-1 else " "}}}\n'
    hdr += '};\n'
    
    with open(f'{OUT}/table_{topo_name}_{traffic}.h', 'w') as f:
        f.write(hdr)
    
    yx = np.sum(table)
    print(f'  {topo_name}/{traffic}: {yx} YX / {num_nodes*num_nodes-num_nodes} pairs ({100*yx/(num_nodes*num_nodes-num_nodes):.0f}%)')

print(f'\n✅ Routing tables saved to {OUT}/')
print('\n📋 Usage: Copy .h files to booksim2/src/ and recompile')